In [95]:
import json
import requests
import numpy as np
import pandas as pd

In [28]:
with open('../key_ring.json') as fi:
    credentials = json.load(fi)

In [29]:
api_key = credentials['Census_ACS']

In [106]:
##endpoint = 'https://api.census.gov/data/2024/acs/acs5/profile?' ## notice the year

In [65]:
years = np.arange(2009, 2025)

## DP05  
I'll use the ACS Demographic & Housing Estimates to get total population estimates for each state.  

* DP05_0001E: total population estimate
* DP05_0001M: margin of error for total population estimate


In [108]:
DP05_params = {
    'get': 'NAME,GEO_ID,DP05_0001E,DP05_0001M', ## field names, separated by comma
    'for': 'state:*',
    'key': api_key
}

In [126]:
DP05_df = pd.DataFrame()

In [129]:
for year in years:
    endpoint = 'https://api.census.gov/data/' + str(year) +'/acs/acs5/profile?'
    response = requests.get(endpoint, params = DP05_params).json()
    df = pd.DataFrame(response)
    df.columns = df.iloc[0]
    df = df[1:]
    df['year'] = year

    DP05_df = pd.concat([DP05_df, df])

DP05_df

,NAME,GEO_ID,DP05_0001E,DP05_0001M,state,year
1,Alabama,0400000US01,4633360,0,01,2009
2,Alaska,0400000US02,683142,0,02,2009
3,Arizona,0400000US04,6324865,0,04,2009
4,Arkansas,0400000US05,2838143,0,05,2009
5,California,0400000US06,36308527,0,06,2009
...,...,...,...,...,...,...
48,Washington,0400000US53,7816116,-555555555,53,2024
49,West Virginia,0400000US54,1778373,-555555555,54,2024
50,Wisconsin,0400000US55,5914872,-555555555,55,2024
51,Wyoming,0400000US56,582397,-555555555,56,2024


In [130]:
DP05_df = DP05_df.rename(columns = {'DP05_0001E': 'pop_est', 'DP05_0001M': 'pop_est_me'})

In [131]:
DP05_df

,NAME,GEO_ID,pop_est,pop_est_me,state,year
1,Alabama,0400000US01,4633360,0,01,2009
2,Alaska,0400000US02,683142,0,02,2009
3,Arizona,0400000US04,6324865,0,04,2009
4,Arkansas,0400000US05,2838143,0,05,2009
5,California,0400000US06,36308527,0,06,2009
...,...,...,...,...,...,...
48,Washington,0400000US53,7816116,-555555555,53,2024
49,West Virginia,0400000US54,1778373,-555555555,54,2024
50,Wisconsin,0400000US55,5914872,-555555555,55,2024
51,Wyoming,0400000US56,582397,-555555555,56,2024


In [133]:
DP05_df['pop_est_me'].value_counts()

pop_est_me
-555555555    832
0             156
Name: count, dtype: int64

Looks like there might be an issue with the way the margin of error is coming through... I'm not sure how much I'll actually be using those, so I'll leave it for now and maybe dig in to it later

In [132]:
DP05_df.to_csv('../Data/DP05.csv')

## DP03  